# planetesimal_sim on Google ColabRun the cells **in order**. Sections 1-4 take about 10 minutes total andreproduce every number in Chapter III. Section 5 is the production run andtakes hours, so it is written to survive Colab disconnecting on you.**Before you start:** Runtime -> Change runtime type -> **CPU**. A GPU willnot help. This code uses numba on the CPU; a GPU runtime just wastes yourquota.

## 1. Install and check the machine

In [ ]:
!pip install -q pytreegravimport os, platform, numpy, scipyprint("CPUs available :", os.cpu_count())print("platform       :", platform.platform())print("numpy / scipy  :", numpy.__version__, "/", scipy.__version__)

Note the CPU count. Colab's free tier usually gives 2 vCPUs, which iscomparable to a laptop, not dramatically better. Colab's advantage here isthat it does not tie up your own machine, not that it is fast.

## 2. Get the code onto the VMPick ONE of the two cells below.**Option A (recommended):** upload the `planetesimal_sim_thesis` folder toyour Google Drive, then mount Drive. Your results survive when the sessiondies.**Option B:** upload the zip directly. Faster to set up, but everything islost when the session ends.

In [ ]:
# OPTION A -- Google Drive (recommended)from google.colab import drivedrive.mount('/content/drive')# Change this to wherever you put the folder in your Drive:PROJECT = '/content/drive/MyDrive/planetesimal_sim_thesis'%cd $PROJECT!ls

In [ ]:
# OPTION B -- upload a zip instead (session-only, results are lost on disconnect)# from google.colab import files# up = files.upload()          # choose planetesimal_sim_thesis.zip# !unzip -q -o planetesimal_sim_thesis.zip -d /content# PROJECT = '/content/planetesimal_sim_thesis'# %cd $PROJECT

## 3. Validation tests -- proves the physics is right (~1 min)

In [ ]:
!python tests/test_validation.py

All eight should pass. If any fails, stop and do not run anything else --something is wrong with the install, not with your thesis.

## 4. Benchmarks -- these produce your Chapter III tables (~6 min)

In [ ]:
!python benchmarks.py --quick        # B1, B2, B3, B5 -- about 40 seconds

In [ ]:
!python benchmarks.py                # adds B4 convergence -- about 6 minutes

Save `benchmarks.json` and the printed output. **These are your numbers, fromyour machine.** They are what belongs in the thesis, not the reference valuesin the README.

## 5. Feasibility check before committing hours of compute

In [ ]:
!python run_scenarios.py --all --dry-run

Read the warnings. Then check the timing on a short run before starting thereal thing:

In [ ]:
import timet0 = time.time()!python run_scenarios.py --scenario S1_gravity_only --n 2000 --t-end 20 \    --spo 128 --snapshot-every 5 --dump-every 1e9 --outdir /content/timing_testelapsed = time.time() - t0print(f"\n20 simulated yr took {elapsed:.0f} s wall clock.")print(f"Extrapolated to 20,000 yr: {elapsed*1000/3600:.1f} hours per scenario.")print(f"All four scenarios: {elapsed*4000/3600:.1f} hours.")

The first ~30 seconds of any run is numba compiling, so this extrapolation ispessimistic — but use it as your upper bound. If four scenarios comes to morethan about 30 hours, reduce `--t-end` or raise `--f` before you start.

## 6. Production runs, written to survive disconnectionColab free tier kills sessions after roughly 12 hours, and sooner if you closethe tab. The `--resume` flag handles this: the run checkpoints itself every250 simulated years, and re-running the identical command picks up exactlywhere it stopped.**Write results to Drive** (`--outdir` under `/content/drive/...`) or thecheckpoints die with the session and resume has nothing to resume from.Run one scenario per cell. If a session dies, just re-run the same cell.

In [ ]:
OUT = '/content/drive/MyDrive/thesis_results'   # must be on Drive!python run_scenarios.py --scenario S1_gravity_only --n 2000 --t-end 2e4 \    --spo 128 --outdir $OUT --resume

In [ ]:
!python run_scenarios.py --scenario S2_gas --n 2000 --t-end 2e4 \    --spo 128 --outdir $OUT --resume

In [ ]:
!python run_scenarios.py --scenario S3_planet --n 2000 --t-end 2e4 \    --spo 128 --outdir $OUT --resume

In [ ]:
!python run_scenarios.py --scenario S4_gas_planet --n 2000 --t-end 2e4 \    --spo 128 --outdir $OUT --resume

### Optional fifth run: the enhanced-drag parameter studyNominal gas drag cannot act on 438 km bodies within 20,000 yr (benchmark B5).This run compresses the drag timescale into the integration window so you canshow what drag *would* do, clearly labelled as a parameter study rather thanthe nominal nebula.

In [ ]:
!python run_scenarios.py --scenario S2_gas --n 2000 --t-end 2e4 --spo 128 \    --gas-boost 20 --outdir ${OUT}_gasboost20 --resume

## 7. Figures and the comparison table

In [ ]:
!python -m planetesimal_sim.analysis $OUT

In [ ]:
from IPython.display import Image, displayimport globfor f in sorted(glob.glob(f'{OUT}/figures/*.png')):    print(f)    display(Image(f))

## 8. Keeping the session aliveColab disconnects idle tabs. Options, in order of honesty:1. **Just let it disconnect and re-run the cell.** With `--resume` you lose at   most 250 simulated years. This is the approach I would use.2. **Colab Pro** gives longer sessions and better CPUs. If your runs are   taking 6+ hours each, it is probably worth the cost for a few weeks.3. Keep the browser tab open and the laptop awake. Unreliable.Do **not** use the JavaScript auto-clicker tricks you will find on StackOverflow. They violate Colab's terms and can get your account limited, whichwould be a bad week to have during a thesis.